# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 14: Beschäftigung (Gegenstück zur Arbeitslosigkeit)

Letzter offener Punkt aus [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md): Das Blatt
"1.1 Eckwerte" enthält — unterhalb der bisher genutzten Zeilen 9–47 — noch einen weiteren,
eigenständigen Block: eine **quartalsweise Beschäftigtenstatistik** (sozialversicherungspflichtig
und ausschließlich geringfügig entlohnte Beschäftigung, jeweils nach **Arbeitsort (AO)** und
**Wohnort (WO)**) — das Gegenstück zur Arbeitslosigkeit: nicht wer arbeitslos ist, sondern wer
beschäftigt ist.

**Arbeitsort vs. Wohnort:** AO zählt Beschäftigungsverhältnisse, deren **Arbeitsplatz** in
Regensburg liegt; WO zählt Personen, die in Regensburg **wohnen**, unabhängig vom Arbeitsort.
Die Differenz WO − AO ist der **Pendlersaldo**: wie viele Regensburger mehr auswärts arbeiten,
als von auswärts nach Regensburg einpendeln.

⚠️ **Wichtige Einschränkung — abweichender Rhythmus und Zeitraum:** Anders als alle bisherigen
Kennzahlen ist dieser Block **quartalsweise** (nicht monatlich) und hat laut Fußnote im Bericht
eine **Wartezeit von 6 Monaten**. In den 18 Monatsberichten (Januar 2025 – Juni 2026) tauchen
dadurch nur **7 unterschiedliche Quartale** auf, jeweils identisch über 3 aufeinanderfolgende
Monatsberichte hinweg gemeldet — und das abgedeckte Zeitfenster liegt entsprechend früher:
**Juni 2024 – Dezember 2025**, statt Januar 2025 – Juni 2026 wie in den übrigen Kapiteln.

In [1]:
from pathlib import Path
import re

import openpyxl
import pandas as pd
import plotly.graph_objects as go

DATA_DIR = Path("..") / "Data" / "RegensburgJCData"
PROCESSED_DIR = Path("..") / "Data" / "processed"
ABBILDUNGEN_DIR = Path("..") / "Docs" / "Abbildungen"

pd.set_option("display.max_columns", None)


def parse_beschaeftigung(path: Path) -> dict:
    '''Liest die quartalsweise Beschaeftigtenstatistik aus "1.1 Eckwerte"
    (Zeilen 49-58: Merkmale-Kopf mit Quartalsdatum, danach vier Kennzahlen
    jeweils fuer den aktuellsten gemeldeten Quartalswert in Spalte B).
    '''
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["1.1 Eckwerte"]
    return {
        "bericht": path.name,
        "quartal": pd.Timestamp(ws.cell(row=49, column=2).value),
        "ao_svp": ws.cell(row=55, column=2).value,
        "wo_svp": ws.cell(row=56, column=2).value,
        "ao_geringfuegig": ws.cell(row=57, column=2).value,
        "wo_geringfuegig": ws.cell(row=58, column=2).value,
    }


files = sorted(DATA_DIR.glob("*.xlsx"))
records = [parse_beschaeftigung(f) for f in files]
alle_berichte = pd.DataFrame(records)

# Da derselbe Quartalswert ueber 3 aufeinanderfolgende Monatsberichte hinweg
# identisch gemeldet wird, auf die tatsaechlich eindeutigen Quartale reduzieren.
beschaeftigung = (
    alle_berichte.drop_duplicates(subset=["quartal"])
    .sort_values("quartal")
    .set_index("quartal")
    .drop(columns=["bericht"])
)
print(f"{len(alle_berichte)} Berichte gelesen, davon {len(beschaeftigung)} eindeutige Quartale.")
beschaeftigung

C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:33: UserWarning: DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.
  warn("DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.")
C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


18 Berichte gelesen, davon 7 eindeutige Quartale.


,ao_svp,wo_svp,ao_geringfuegig,wo_geringfuegig
quartal,,,,
2024-06-01,54214,88011,8310,10668
2024-09-01,55037,89093,8184,10489
2024-12-01,54495,88564,8234,10507
2025-03-01,54421,88326,8173,10480
2025-06-01,54513,88252,8286,10721
2025-09-01,55331,89259,8150,10448
2025-12-01,54653,88662,8140,10405


## 1. Sozialversicherungspflichtige Beschäftigung: Arbeitsort vs. Wohnort

Wie entwickeln sich die beiden Perspektiven — Jobs in Regensburg (AO) vs. beschäftigte
Regensburger (WO) — über die 7 verfügbaren Quartale?

In [2]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=beschaeftigung.index, y=beschaeftigung["ao_svp"], name="Arbeitsort (AO)",
    mode="lines+markers", line=dict(color="#2563eb", width=2.5), marker=dict(size=7),
))
fig.add_trace(go.Scatter(
    x=beschaeftigung.index, y=beschaeftigung["wo_svp"], name="Wohnort (WO)",
    mode="lines+markers", line=dict(color="#7c3aed", width=2.5), marker=dict(size=7),
))
fig.update_layout(
    title="Sozialversicherungspflichtig Beschäftigte: Arbeitsort vs. Wohnort",
    xaxis_title="Quartal", yaxis_title="Personen", template="plotly_white",
    height=450, hovermode="x unified", legend=dict(orientation="h", y=1.15),
)
fig.write_image(ABBILDUNGEN_DIR / "abb41_beschaeftigung_ao_wo.png", scale=2)
fig

**Befund:** Beide Perspektiven sind über die 7 Quartale hinweg bemerkenswert stabil und bewegen sich fast parallel: Arbeitsort schwankt zwischen 54.214 und 55.331, Wohnort zwischen 88.011 und 89.259. Der Wohnort-Wert liegt durchgehend rund 34.000 Personen über dem Arbeitsort-Wert — mehr dazu im Pendlersaldo unten.

## 2. Pendlersaldo: wie viele Regensburger mehr auswärts arbeiten

Die Differenz WO − AO zeigt den Pendlersaldo direkt als eigene Kennzahl.

In [3]:
pendlersaldo = beschaeftigung["wo_svp"] - beschaeftigung["ao_svp"]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=beschaeftigung.index, y=pendlersaldo, marker_color="#059669", opacity=0.8,
    name="Pendlersaldo (WO − AO)",
))
fig.update_layout(
    title="Pendlersaldo: Auspendlerüberschuss (WO − AO, sozialversicherungspflichtig)",
    xaxis_title="Quartal", yaxis_title="Personen", template="plotly_white", height=420, yaxis=dict(range=[33000, 34500]),
    showlegend=False,
)
fig.write_image(ABBILDUNGEN_DIR / "abb42_beschaeftigung_pendlersaldo.png", scale=2)
fig

**Befund:** Der Pendlersaldo ist außergewöhnlich stabil — er bewegt sich nur zwischen 33.739 und 34.069 Personen (Mittelwert ≈ 33.929) über alle 7 Quartale. Es pendeln also durchgehend rund 34.000 mehr sozialversicherungspflichtig Beschäftigte aus Regensburg aus, als von auswärts einpendeln — ein struktureller, kaum schwankender Auspendlerüberschuss.

## 3. Geringfügig entlohnte Beschäftigung: Arbeitsort vs. Wohnort

Dieselbe Arbeitsort/Wohnort-Perspektive für ausschließlich geringfügig entlohnte
Beschäftigung (Minijobs).

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=beschaeftigung.index, y=beschaeftigung["ao_geringfuegig"], name="Arbeitsort (AO)",
    mode="lines+markers", line=dict(color="#f59e0b", width=2.5), marker=dict(size=7),
))
fig.add_trace(go.Scatter(
    x=beschaeftigung.index, y=beschaeftigung["wo_geringfuegig"], name="Wohnort (WO)",
    mode="lines+markers", line=dict(color="#db2777", width=2.5), marker=dict(size=7),
))
fig.update_layout(
    title="Ausschließlich geringfügig entlohnte Beschäftigung: Arbeitsort vs. Wohnort",
    xaxis_title="Quartal", yaxis_title="Personen", template="plotly_white",
    height=450, hovermode="x unified", legend=dict(orientation="h", y=1.15),
)
fig

**Befund:** Die geringfügige Beschäftigung entwickelt sich in beiden Perspektiven leicht rückläufig — Arbeitsort von 8.310 auf 8.140 (−2,0 %), Wohnort von 10.668 auf 10.405 (−2,5 %) — bei ebenfalls stabilem Abstand zwischen beiden Werten.

## 4. Anteil geringfügiger Beschäftigung an der Gesamtbeschäftigung

Wie groß ist der Minijob-Sektor im Verhältnis zur sozialversicherungspflichtigen
Beschäftigung — und verändert sich dieses Verhältnis?

In [5]:
anteil_ao = beschaeftigung["ao_geringfuegig"] / (beschaeftigung["ao_geringfuegig"] + beschaeftigung["ao_svp"]) * 100
anteil_wo = beschaeftigung["wo_geringfuegig"] / (beschaeftigung["wo_geringfuegig"] + beschaeftigung["wo_svp"]) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=beschaeftigung.index, y=anteil_ao, name="Arbeitsort (AO)", mode="lines+markers", line=dict(color="#f59e0b", width=2.5)))
fig.add_trace(go.Scatter(x=beschaeftigung.index, y=anteil_wo, name="Wohnort (WO)", mode="lines+markers", line=dict(color="#db2777", width=2.5)))
fig.update_layout(
    title="Anteil geringfügiger Beschäftigung an der Gesamtbeschäftigung",
    xaxis_title="Quartal", yaxis_title="Anteil (%)", yaxis_ticksuffix="%",
    template="plotly_white", height=420, legend=dict(orientation="h", y=1.15),
)
fig.write_image(ABBILDUNGEN_DIR / "abb43_beschaeftigung_anteil_geringfuegig.png", scale=2)
fig

**Befund:** Der Anteil geringfügiger Beschäftigung sinkt in beiden Perspektiven leicht — am Arbeitsort von 13,3 % auf 13,0 %, am Wohnort von 10,8 % auf 10,5 %. Das passt zur in Kapitel 7 beobachteten leichten Verschiebung zu höheren Einkommen bei den Aufstockern — ein Muster, das sich hier unabhängig auch in der Gesamtbeschäftigtenstatistik zeigt.

## 5. Zusammenfassung

Die quartalsweise Beschäftigtenstatistik zeigt ein neues Bild: Sowohl die sozialversicherungspflichtige als auch die geringfügige Beschäftigung sind über die 7 verfügbaren Quartale (Juni 2024 – Dezember 2025) bemerkenswert stabil. Der auffälligste Befund ist der Pendlersaldo: Mit einem fast unveränderten Überschuss von rund 34.000 auspendelnden gegenüber einpendelnden sozialversicherungspflichtig Beschäftigten zeigt sich eine strukturelle Konstante der lokalen Arbeitsmarktgeografie, die von den Schwankungen der Arbeitslosenzahlen unberührt bleibt. Der leicht sinkende Anteil geringfügiger Beschäftigung bestätigt zudem unabhängig die bereits in Kapitel 7 beobachtete leichte Verschiebung zu höheren Einkommen.

Damit ist die in [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md) priorisierte Themenliste vollständig abgearbeitet.

In [6]:
beschaeftigung.to_csv(PROCESSED_DIR / "beschaeftigung_quartal.csv")
print("Gespeichert:", PROCESSED_DIR / "beschaeftigung_quartal.csv")

Gespeichert: ..\Data\processed\beschaeftigung_quartal.csv
